# Skin Review: patient-friendly melanoma research app

This notebook launches the redesigned software using your **already saved Attention U-Net + EfficientNetB0 models**. It does not train models, tune cutoffs or change their image processing.

**How to run:** choose a GPU runtime if available, then run the four code cells below in order. In the second cell, enter your actual team contact details. Allow the usual Google Drive mount in the third cell. Open the app link printed by the last cell. Keep the runtime connected.

The app includes a clear result, next steps, an expandable pipeline explanation, result interpretation, melanoma information, project contacts, a downloadable doctor PDF, and your saved evaluation tables and curves.

**Research use only.** Use de-identified dermoscopic research images. A public demo link is not a private clinical portal. Results cannot diagnose or rule out melanoma. The user downloads the report; nothing is emailed automatically.


## 1. Install the interface and PDF packages
This keeps the TensorFlow already supplied by Colab.

In [ ]:
%pip -q install "gradio==6.30.0" "reportlab>=4,<5"

## 2. Set your project details
Edit only the quoted fields. Leave `RUN_DIR` empty to find your existing saved run automatically. Contact fields appear in the app and doctor report.

In [ ]:
PROJECT_NAME = "Skin Review"
TEAM_NAME = "Melanoma Image AI project team"
TEAM_EMAIL = ""       # Replace with your real team email
PROJECT_WEBSITE = ""  # Optional HTTPS project website
RUN_DIR = ""          # Optional: paste a complete saved model/reporting run folder


## 3. Load the app and your saved models
Allow Google Drive access when Colab asks. If the saved folder has moved, paste its new path into `RUN_DIR` above and run the setup and load cells again. The required files are listed in a helpful error if they cannot be found.

In [ ]:
from google.colab import drive
from pathlib import Path
import base64, hashlib, io, sys, zipfile

drive.mount("/content/drive")
APP_DIR = Path("/content/skin_review_app")
APP_DIR.mkdir(parents=True, exist_ok=True)
_EMBEDDED_SOURCE = "..."
_SOURCE_SHA256 = "64b2d8d15806a9c11ad2889cf9ac50e52ed4250c71c90caedc8c23565f6d61bb"
_source = base64.b64decode(_EMBEDDED_SOURCE)
if hashlib.sha256(_source).hexdigest() != _SOURCE_SHA256:
    raise RuntimeError("The embedded app source is damaged. Open a fresh copy of this notebook.")
with zipfile.ZipFile(io.BytesIO(_source)) as archive:
    for item in archive.infolist():
        target = (APP_DIR / item.filename).resolve()
        if not target.is_relative_to(APP_DIR.resolve()):
            raise RuntimeError("Unexpected path in the embedded app source.")
    archive.extractall(APP_DIR)
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

from attention_pipeline_core import MelanomaPipeline
from settings import AppSettings
from skin_review_ui import launch_app

_REQUIRED = ["models/attention_unet_best.keras", "models/selected_final_classifier.keras",
             "tables/selected_configuration.json"]
def complete(path):
    return all((path / filename).is_file() for filename in _REQUIRED)

if RUN_DIR.strip():
    chosen_run = Path(RUN_DIR.strip())
else:
    preferred = Path("/content/drive/MyDrive/Melanoma_Attention_UNet_Reporting_and_App")
    original = Path("/content/drive/MyDrive/Melanoma_ISIC2016_Final_Candidate")
    candidates = [preferred / "20261012T051542_296259Z"]
    if preferred.is_dir():
        candidates += sorted((p for p in preferred.iterdir() if p.is_dir()), reverse=True)
    candidates += [original / "20261007T163637_386595Z"]
    if original.is_dir():
        candidates += sorted((p for p in original.iterdir() if p.is_dir()), reverse=True)
    chosen_run = next((p for p in candidates if complete(p)), None)
if chosen_run is None or not complete(chosen_run):
    raise FileNotFoundError("The saved model run could not be found. Set RUN_DIR in cell 2 to the folder containing:\n" + "\n".join(_REQUIRED))

print("Loading your frozen models from:", chosen_run)
pipeline = MelanomaPipeline.load(chosen_run)
settings = AppSettings(project_name=PROJECT_NAME, team_name=TEAM_NAME,
                       team_email=TEAM_EMAIL, project_website=PROJECT_WEBSITE)
sample_paths = sorted((APP_DIR / "examples").glob("*.jpg"))
print("Models ready. Saved classification cutoff:", pipeline.configuration["threshold"])
print("No training or threshold tuning has been performed. Run the final cell to open the app.")


## 4. Open the redesigned app
Use public research images for your presentation. In the app, open **Prepare a report for your doctor**, add optional notes and select **Create doctor report** to download a PDF. It is not sent automatically.

In [ ]:
# Closing a previous instance makes this cell safe to rerun after editing settings.
if "demo" in globals():
    demo.close()
demo = launch_app(pipeline, settings, sample_paths, share=True, debug=False)


### If you need help
- **No models found:** set `RUN_DIR` to your completed reporting run in Drive, then rerun cells 2-4.
- **Contact details are blank:** enter the real email/name in cell 2 and rerun cells 2-4.
- **Evaluation section is empty:** choose the completed reporting run; the original frozen model run may not include the report tables or plots.
- **The app link stops:** reconnect the runtime, load the models and launch again. Gradio links are temporary.
- **The spot concerns you:** arrange a clinical skin check regardless of the model output. Healthdirect Australia: 1800 022 222. Cancer Council / CancerConnect: 13 11 20.

Patient information: [Cancer Council](https://connect.cancer.org.au/cancer-information/types-of-cancer/melanoma) and [Healthdirect](https://www.healthdirect.gov.au/contact-us). Source links checked 16 October 2026.
